# Домашнее задание по теме «Метрические и линейные алгоритмы»



Для большинства алгоритмов классического машинного обучения давно существуют готовые реализации: мы уже использовали kNN из sklearn в семинарах. Это очень удобно для решения практических задач, потому что для получения результата достаточно нескольких строк кода. Но чтобы эффективно использовать такие модели, важно понимать их ограничения, сильные и слабые стороны — то, как они работают «под капотом».

В этом домашнем задании ты узнаешь детали реализации метрических алгоритмов:
- kNN
- Линейная регрессия

Как понять, что задание выполнено хорошо:

- все `# Напиши код здесь` заполнены, и код работает без ошибок;
- результаты твоих реализаций KNNClassifier и KNNRegressor точно совпадают с результатами из sklearn;
- даны чёткие, развёрнутые ответы на вопросы.

# Метрические модели

## Подготовка

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report,
    mean_squared_error, r2_score
)

In [4]:
def get_train_test(complex_function:callable, num_points=100, outliers=False, **kwargs):
    """ Генерация данных """
    np.random.seed(123)

    X = np.linspace(0, 10, num_points)
    X = X.reshape(-1, 1)

    X_train = X[::3] # Берём каждый третий элемент для обучения
    X_test = X[np.setdiff1d(np.arange(len(X)), np.arange(0, len(X), 3))] # Оставшиеся — для теста
    # Тут мы могли бы сгенерировать новый х: например, более плотно распределённые точки
    # Главное — что есть некоторые точки исходной функции, между которыми нужно заполнить пустое пространство

    # Значения функции с шумом
    y_train = complex_function(X_train).ravel() + np.random.normal(0, 0.2, len(X_train))
    y_test = complex_function(X_test).ravel() + np.random.normal(0, 0.2, len(X_test))

    if outliers:
        y_train = add_outliers(y_train, **kwargs)

    return X, X_train, X_test, y_train, y_test

def add_outliers(y,
                 num_outliers=None,
                 strength_std_devs=None,
                 random_seed=42):
    """ Добавление выбросов """

    if not num_outliers:
        num_outliers = int(len(y_train) * 0.1)

        if num_outliers == 0:
            num_outliers = 2

    if not strength_std_devs:
        strength_std_devs = 5

    y = np.array(y)
    y_with_outliers = y.copy()

    std_dev = np.std(y)

    if std_dev == 0:
        outlier_shift = strength_std_devs
    else:
        outlier_shift = strength_std_devs * std_dev

    rng = np.random.RandomState(seed=random_seed)

    # Случайно выбираем индексы выбросов
    outlier_indices = rng.choice(len(y), num_outliers, replace=False)

    for idx in outlier_indices:
        # Произвольно решаем, будет ли outlier больше или меньше текущего значения
        sign = rng.choice([-1, 1])
        y_with_outliers[idx] += sign * outlier_shift * (rng.rand() * 0.5 + 0.5)

    return y_with_outliers

def plot_data(X, X_train, X_test, y_train, y_test,
              complex_function:callable,
              pred=None,
              labels=[''],
              title='Синтетические данные для регрессии',
              subplot=None,
              highlight_train_range=None):
    """ Визуализация данных/результатов"""

    if not subplot:
        plt.figure(figsize=(12, 6))

    plt.scatter(X_train, y_train, c='black', alpha=0.5, label='Обучающая выборка')
    plt.scatter(X_test, y_test, c='gray', alpha=0.5, label='Тестовая выборка')
    plt.plot(X, complex_function(X), 'b-', label='Истинная функция')

    if pred is not None:
        for y_pred, label in zip(pred, labels):
            plt.plot(X_test, y_pred, alpha=1, label=label)

    if highlight_train_range:
        plt.axvspan(highlight_train_range[0], highlight_train_range[1],
                     color='lightgray', alpha=0.4, zorder=-1, label='Диапазон обучения')

    plt.grid(True)
    plt.legend()
    plt.title(title)

## Реализация собственного kNN

### Задача 1. Реализация BaseKNN [2 балла]

Класс ```BaseKNN``` — основа всех моделей. Его задачи:
* хранение обучающей выборки;
* реализация логики ключевого функционала kNN — поиска ближайших соседей (метод ```_find_k_nearest_neighbours```);
* вычисление весов на основе расстояний (метод ```_calculate_weights```).

Ниже представлен код с базовой структурой классов. Твоя задача — заполнить пропущенные части, руководствуясь комментариями и описаниями.

#### Задача 1.1. Функции ядра [0,5 балла]

Реализуй следующие функции ядра:

1. Гауссово — `_gaussian_kernel`:
$$\frac{1}{\sqrt{2\pi}}\exp(-\frac{1}{2}d^2).$$
2. Епанечникова — `_epanechnikov_kernel`:
$$\frac{3}{4}(1-d^2)\cdot I(|d|\le1).$$
3. Треугольное — `_triangular_kernel`:
$$(1-|d|)\cdot I(|d|\le1).$$
4. Прямоугольное — `_uniform_kernel`:
$$\frac{1}{2}I(|d|\le1).$$

#### Задача 1.2. Метод ```fit``` [0,5 балла]

1. Преобразуй $X$ и $Y$ в `np.array` с помощью функции `ensure_numpy`.
2. Убедись, что $X$ — двумерный массив (`n_points`, `n_features`).
3. Если $X$ одномерный (`n_points`), преобразуй его в (`n_samples`, 1): это необходимо для корректного вычисления расстояний.

#### Задача 1.3. Метод ```_find_k_nearest_neighbours``` [0,5 балла]

1. Вычисли расстояния от $x$ до всех точек в `self.X_train`, используя `np.linalg.norm (..., $axis=1$)`.
2. Найди индексы ближайших соседей, используя `np.argsort()`.
3. Найди расстояния до этих ближайших соседей.

#### Задача 1.4. Метод ```_calculate_weights``` [0,5 балла]

Примени взвешивание `inverse distance` для `weights == 'distance'`:
$$\frac{1}{\left|d\right|+\epsilon}.$$

В ином случае:
1. Масштабируй расстояния на `self.bandwidth` и `self.epsilon`.
2. Примени выбранную функцию ядра `self._weighting_function`.

> **Примечание.** Здесь $d$ — distances (`np.array`).

In [5]:
from collections import defaultdict, Counter
import logging

def ensure_numpy(data):
    """ Функция для проверки типа входных данных """
    if isinstance(data, pd.DataFrame) or isinstance(data, pd.Series):
        return data.values
    return np.asarray(data)

class BaseKNN():
    def __init__(self, n_neighbours=5, weights='uniform', kernel='uniform', bandwidth=1.0, epsilon=1e-6):
        """ Базовый класс, который отвечает за запоминание выборки и поиск ближайших соседей """
        self.n_neighbours = n_neighbours
        self.weights = weights
        self.kernel = kernel
        self.bandwidth = bandwidth
        self.epsilon = epsilon
        self.X_train = None
        self.y_train = None

        self._weighting_function = self._get_kernel_function(self.kernel)

    def _gaussian_kernel(self, d):
        # Напиши код здесь
        return (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * d**2)

    def _epanechnikov_kernel(self, d):
        # Напиши код здесь
        a = 0.75 * (1 - d**2)
        a[np.abs(d) > 1] = 0
        return a

    def _triangular_kernel(self, d):
        # Напиши код здесь
        a = 1 - np.abs(d)
        a[np.abs(d) > 1] = 0
        return a

    def _uniform_kernel(self, d):
        # Напиши код здесь
        a = np.full_like(d, 0.5)
        a[np.abs(d) > 1] = 0
        return a

    def _get_kernel_function(self, kernel_name):
        """
        Возвращает функцию ядра в зависимости от выбранного типа
        """
        kernels = {
            'gaussian': self._gaussian_kernel,
            'epanechnikov': self._epanechnikov_kernel,
            'triangular': self._triangular_kernel,
            'uniform': self._uniform_kernel
        }

        return kernels.get(kernel_name)

    def fit(self, X, y):
        """
        Обучение kNN. Фактически - запоминание данных из обучающей выборки
        """
        self.X_train = ensure_numpy(X)
        self.y_train = ensure_numpy(y)
        if self.X_train.ndim == 1:
            self.X_train = self.X_train.reshape(-1, 1)
        # Напиши код здесь: запомни обучающие данные X и y в self.X_train и self.y_train
        return self

    def _find_k_nearest_neighbours(self, x):
        """
        Поиск k ближайших соседей для одной точки

        :param x: одна точка из тестовой выборки, shape (n_features)

        :return k_indices: индексы ближайших соседей из обучающей выборки
        :return k_distances: расстояния до каждого из k ближайших соседей
        """
        # Напиши код здесь: реализуй поиск k ближайших соседей
        distances = np.linalg.norm(self.X_train - x, axis=1)# Напиши код здесь
        k_indices = np.argsort(distances)[:self.n_neighbours]# Напиши код здесь
        k_distances = distances[k_indices]# Напиши код здесь
        return k_indices, k_distances


    def _calculate_weights(self, distances):
        """
        Подсчёт весов для одной точки. Масштабируем расстояния, используя
        параметр bandwidth, затем передаём их в функцию ядра.

        :param distances: массив расстояний от тестовой точки до других точек (n_features)

        :return: массив расстояний с применением функции ядра (n_features)
        """
        if self.weights == 'distance':
            weights = 1 / (np.abs(distances) + self.epsilon)# Напиши код здесь
            return weights

        scaled_distances = distances / (self.bandwidth + self.epsilon)# Напиши код здесь
        weighted_distances = self._weighting_function(scaled_distances)# Напиши код здесь
        return weighted_distances


    def predict(self, X):
        """ Реализуем этот метод в дочерних классах """
        raise NotImplementedError("Не реализован в базовом kNN")


Основа готова. В ней реализован весь базовый функционал, необходимый kNN для решения любой задачи.

### Задача 2. KNNRegressor [3 балла]

Займёмся регрессией. Идея в следующем: предсказание новой точки — это среднее (или взвешенное среднее) значений её соседей.

Нужно:
 * реализовать обычный регрессор (```if self.weights == 'uniform':```),
 * протестировать его,
 * перейти к взвешенному.

> **Примечание.** Далее все указания будут приведены в комментариях.

#### Задача 2.1. Метод ```predict``` [1,5 балла]

Реализуй метод ```predict``` для регрессора.

#### Задача 2.2. Метод ```predict``` (взвешивание) [1,5 балла]

Реализуй взвешивание в методе  ```predict```: `if self.weights != 'uniform'`.

In [6]:
class KNNRegressor(BaseKNN):
    def __init__(self, n_neighbours=5, weights='uniform', kernel='uniform', bandwidth=1.0, epsilon=1e-6):
        super().__init__(n_neighbours=n_neighbours,
                         weights=weights,
                         kernel=kernel,
                         bandwidth=bandwidth,
                         epsilon=epsilon)

    def predict(self, X):
        """
        Предсказывает значения для тестовой выборки X

        :param X: тестовые данные (n_test_points, n_features)

        :return y_pred: массив предсказанных для каждой точки значений (n_test_points)
        """

        # Преобразуй X в numpy.array по аналогии с методом .fit()
        # Убедись, что X — двумерный массив

        X = ensure_numpy(X)# Напиши код здесь
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        # Инициализируй массив для предсказаний (n_test_points)
        # Значения в нём будут обновляться итеративно
        y_pred = np.zeros(X.shape[0])# Напиши код здесь

        for i, x in enumerate(X):
            k_indices, k_distances = self._find_k_nearest_neighbours(x)# Напиши код здесь: индексы и расстояния k ближайших соседей
            k_targets = self.y_train[k_indices]# Напиши код здесь: значения целевой переменной для этих соседей

            if self.weights == 'uniform':
                # Это обычный kNN. Предсказание — среднее значение k_targets (или медиана)
                # Напиши код здесь
                y_pred[i] = np.mean(k_targets)
            else:
                # Взвешенный kNN. Предсказание — взвешенное среднее k_targets
                # Напиши код здесь
                weights = self._calculate_weights(k_distances)
                y_pred[i] = np.sum(weights * k_targets) / np.sum(weights)
        return y_pred



Отлично! Теперь попробуй протестировать свою модель. Результаты должны сойтись с sklearn.

In [7]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='target')


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_reg = KNNRegressor(n_neighbours=5)
knn_reg_Sklearn = KNeighborsRegressor(n_neighbors=5)

knn_reg.fit(X_train_scaled, y_train)
knn_reg_Sklearn.fit(X_train_scaled, y_train)

y_pred_reg = knn_reg.predict(X_test_scaled)
y_pred_reg_Sklearn = knn_reg_Sklearn.predict(X_test_scaled)

print(f"Наша реализация R2 Score: {r2_score(y_test, y_pred_reg):.3f}, MSE: {mean_squared_error(y_test, y_pred_reg):.3f}")
print(f"Sklearn R2 Score: {r2_score(y_test, y_pred_reg_Sklearn):.3f}, MSE: {mean_squared_error(y_test, y_pred_reg_Sklearn):.3f}")

Наша реализация R2 Score: 0.670, MSE: 0.432
Sklearn R2 Score: 0.670, MSE: 0.432


In [8]:
same = 0
different = 0
for i, j in zip(y_pred_reg, y_pred_reg_Sklearn):
    if i == j:
        same += 1
    else:
        different += 1
        print('different')

print(f'Совпадений: {same}')
print(f'Различий: {different}')

Совпадений: 4128
Различий: 0


Теперь взвешивания.

In [9]:
knn_reg = KNNRegressor(n_neighbours=5, weights='distance')
knn_reg_Sklearn = KNeighborsRegressor(n_neighbors=5, weights='distance')

knn_reg.fit(X_train_scaled, y_train)
knn_reg_Sklearn.fit(X_train_scaled, y_train)

y_pred_reg = knn_reg.predict(X_test_scaled)
y_pred_reg_Sklearn = knn_reg_Sklearn.predict(X_test_scaled)

print(f"Наша реализация R2 Score: {r2_score(y_test, y_pred_reg):.3f}, MSE: {mean_squared_error(y_test, y_pred_reg):.3f}")
print(f"Sklearn R2 Score: {r2_score(y_test, y_pred_reg_Sklearn):.3f}, MSE: {mean_squared_error(y_test, y_pred_reg_Sklearn):.3f}")

Наша реализация R2 Score: 0.672, MSE: 0.430
Sklearn R2 Score: 0.672, MSE: 0.430


# Линейные модели

В этом задании ты обучишь линейную регрессию, применишь регуляризацию, нормировку, проанализируешь качество модели, а также реализуешь градиентный спуск.

## Подготовка

In [10]:
import gc

import pandas as pd
import numpy as np

import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import time
from tqdm import tqdm

from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler

##  Задача 1 [0,5 балла]

Объясни, почему при использовании регуляризации штрафуются только веса, но не свободный коэффициент $w_0$ (другое обозначение — $b$)?

Напомним, что предсказание линейной модели считается по формуле:

$$a(x) = \langle w, x \rangle + w_0.$$

**Напиши ответ здесь:**

Регуляризация штрафует только веса, потому что именно они определяют сложность модели и её способность переобучаться на шум признаков. w0 лишь смещает предсказания.

-------

Для следующих заданий возьми данные из датасета [Predict Calorie Expenditure](https://www.kaggle.com/competitions/playground-series-s5e5/overview).

Цель соревнования — предсказать, сколько калорий потратит человек за тренировку по признаковому описанию как человека, так и самой тренировки.

In [11]:
! gdown 1jb1tbxmGcgFk-jzNgu-sHXCUkgtAslRq

Downloading...
From: https://drive.google.com/uc?id=1jb1tbxmGcgFk-jzNgu-sHXCUkgtAslRq
To: /content/train.csv
100% 35.5M/35.5M [00:00<00:00, 61.1MB/s]


In [12]:
data = pd.read_csv("/content/train.csv", index_col=0)
data.head()

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
id,,,,,,,,
0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


В этом датасете 8 переменных:
* 1 категориальная;
* 7 вещественных.

`Calories` — целевая переменная.

Раздели датасет на признаки и целевую переменную, а также на `train` и `val`.

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(data.drop('Calories', axis=1), data['Calories'], test_size=0.25, random_state=123)

## Задача 2. OneHotEncoder [0,5 балла]

В семинаре мы использовали `Target Encoding` для приведения категориальных переменных к числовому виду, так как в задаче было очень много категорий. Но для линейных моделей обычно используется `OneHotEncoding`.

Примени [OneHotEncoding из sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) к категориальной переменной и получи новый датасет, который уже будет использоваться для дальнейшего обучения моделей. Но не делай этого в явном виде. Задай последовательность действий в виде [пайплайна из sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html). Он тебе пригодится в задачах.

Затраты по калориям не бывают отрицательными. Модель этого не знает, так как она предсказывает на всей числовой прямой. Поэтому нужно избежать отрицательных значений на этапе вызова `predict` с помощью [TransformedTargetRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html).

Сделать неотрицательными предсказания калорий можно по-разному: округлить до нуля, взять минимальное по обучающей выборке.

**Задание**:
1. Преобразуй датасет к числовому виду.
2. Обучи модель линейной регрессии и сделай предсказание на валидации, чтобы проверить, что всё работает.
3. Добавь постобработку ответов.
4. Оберни все в Pipeline.

In [14]:
# Импортируем нужные модули
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline

# Найди минимальный y
# Напиши код здесь
min_y = y_train.min()
# Внутри Pipeline преобразуй категориальные переменные и передай результат линейной модели
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
], remainder="passthrough")
pipe = Pipeline([
      ('preprocessor', preprocessor),
      ('regressor', TransformedTargetRegressor(
        regressor=LinearRegression(),
        func=lambda y: np.clip(y, min_y, None),
        inverse_func=lambda y: y))
      ])

# Обучи модель и получи предсказания для валидации, чтобы проверить работоспособность
# Напиши код здесь
pipe.fit(X_train, y_train)
y_pred_val = pipe.predict(X_val)
y_pred_val.shape, (y_pred_val < 0).sum()

((187500,), np.int64(8735))

##  Задача 3. Добавь функцию оценки качества работы [1 балл]

В соревновании используется не самая стандартная метрика RMSLE (англ. Root Mean Squared Logarithmic Error).

$$RMSLE = \Big(\frac{1}{n}\sum_{i=1}^n(\log{(1+\hat y_i)} - \log{(1+y_i)})^{2}\Big)^{\frac{1}{2}}$$

Эта метрика есть в реализации [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_log_error.html), но тебе нужно написать её самостоятельно, чтобы в следующих заданиях её можно было передавать внутрь пайплайна через [make_scorer](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html).

Подумай, с каким знаком должна возвращать ответ эта функция для корректной работы `make_scorer`.

**Задание:**
1. Реализуй функцию для подсчёта метрики.
2. Проверь, что она работает корректно (сравни с примером из sklearn).
3. Оцени качество предсказаний на `train` и `val`.
4. Занеси информацию о качестве в таблицу с историей.

In [15]:
from sklearn.metrics import root_mean_squared_log_error

def scoring_foo(y_true, y_pred): # Можешь переименовать. Мы не даём корректный нейминг, потому что он подскажет знак функции
  # Напиши код здесь\
  y_pred = np.clip(y_pred, 0, None)
  y_true = np.clip(y_true, 0, None)
  rmsle = (np.log1p(y_pred) - np.log1p(y_true)) ** 2
  rmsle = np.sqrt(np.mean(rmsle))
  return -rmsle

# Получи предсказания на train
rmsle_scorer = make_scorer(scoring_foo, greater_is_better=False)
y_pred_train = pipe.predict(X_train) # Напиши код здесь
y_val = np.clip(y_val, 0, None)
y_pred_val = np.clip(y_pred_val, 0, None)
# Сравни ответы для валидации
print(scoring_foo(y_val, y_pred_val))
print(root_mean_squared_log_error(y_val, y_pred_val))

# Получи оценку метрики для train и val
rmsle_train = scoring_foo(y_train, y_pred_train)
rmsle_valid = scoring_foo(y_val, y_pred_val)

# Создай контейнер для хранения информации о метриках
res = {}

# Запиши метрики в контейнер и выведи таблицу
res['base'] = [rmsle_train, rmsle_valid]
df_res = pd.DataFrame.from_dict(res, orient='index',
                                columns=['rmse_train', 'rmse_valid'])
df_res

-0.5673800814097045
0.5673800814097045


,rmse_train,rmse_valid
base,-0.564621,-0.56738


##  Задача 4. Выбери модель и нормализацию [1,5 балла]

Используй модели:
* Ridge,
* Lasso,
* ElasticNet.

Для их качественной работы примени нормировки:
* MinMaxScaler,
* StandardScaler,
* RobustScaler.

**Задание.**
Оцени качество работы каждой пары нормировка — модель и занеси результаты в таблицу с соответствующим названием. По названию должно быть понятно, к какой комбинации относится результат.

Нормировку добавь внутрь пайплайна. За основу возьми пайплайн из задания выше.

In [16]:
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler

# Напиши код здесь
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
], remainder='passthrough')

# Контейнер для результатов
res = {}

for model_class, model_name in [(Ridge, 'Ridge'), (Lasso, 'Lasso'), (ElasticNet, 'ElasticNet')]:
    for scaler_class, scaler_name in [(MinMaxScaler, 'MinMax'), (StandardScaler, 'Standard'), (RobustScaler, 'Robust')]:

        pipe = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('scaler', scaler_class()),
            ('regressor', model_class(alpha=1.0, max_iter=1000, random_state=123))
        ])

        pipe.fit(X_train, y_train)

        rmsle_train = scoring_foo(y_train, pipe.predict(X_train))
        rmsle_valid = scoring_foo(y_val, pipe.predict(X_val))

        res[f'{model_name}_{scaler_name}'] = [rmsle_train, rmsle_valid]
df_res = pd.DataFrame.from_dict(res, orient='index',
                                columns=['rmsle_train', 'rmsle_valid'])
# Выведи результат
df_res

,rmsle_train,rmsle_valid
Ridge_MinMax,-0.564590,-0.567345
Ridge_Standard,-0.564620,-0.567380
Ridge_Robust,-0.564620,-0.567379
Lasso_MinMax,-0.469986,-0.468928
Lasso_Standard,-0.573146,-0.573873
Lasso_Robust,-0.576662,-0.577741
ElasticNet_MinMax,-0.898406,-0.898806
ElasticNet_Standard,-0.448930,-0.448114
ElasticNet_Robust,-0.446483,-0.446361


## Задача 5. Добавь полиномы [1,5 балла]



Линейная модель может решать нелинейные задачи, если к признаковому пространству добавить их полиномы. Вряд ли сожжённые за тренировку калории линейно зависят от признаков. Поэтому добавь полиномы различных степеней к признаковому пространству, чтобы улучшить качество решения задачи.

В этом тебе поможет класс [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), он также интегрируется в пайплайн решения. Подумай, куда конкретно его вставлять в пайплайн и нужно ли добавлять полиномы OHE переменных.

Обязательно изучи документацию: там есть один аргумент, который тебе надо выключить.

Для добавления полиномов руководствуйся логикой или общими знаниями. Например, в экономических и социологических исследованиях часто добавляют к данным переменную «квадрат возраста». Её логика в следующем: человек растёт и развивается до определённого возраста, а далее угасает. Скорее всего, целевая переменная и возраст будут зависеть параболически, «квадрат возраста» позволит учесть эту параболическую зависимость.

Если не можешь придумать полиномы, то перебери переменные и значения полиномов для них. Переменных не так много, а модель учится быстро.

**Задание:**
1. Возьми лучшую по валидации комбинацию нормировка — модель.
2. Добавь к пайплайну полиномы для некоторых переменных.
3. Сохрани результат лучшего эксперимента в таблицу (у авторов задания получилось качество 0.238208).
4. Опиши, как выбирал полиномы.

In [31]:
from sklearn.preprocessing import PolynomialFeatures
rmsle_scorer = make_scorer(scoring_foo, greater_is_better=False)


min_y = y_train.min()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ('num', Pipeline([
      ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ]), numerical_features)
], remainder="drop")

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('regressor', TransformedTargetRegressor(
        regressor=Lasso(alpha=0.1),
        func=lambda y: np.clip(y, min_y, None),
        inverse_func=lambda y: y
    ))
])

pipe.fit(X_train, y_train)

y_pred_train = pipe.predict(X_train)
y_pred_val = pipe.predict(X_val)

rmsle_train = -scoring_foo(y_train, y_pred_train)
rmsle_valid = -scoring_foo(y_val, y_pred_val)

print(f"Train RMSLE: {rmsle_train:.6f}")
print(f"Val RMSLE:   {rmsle_valid:.6f}")

res['Ridge_poly2_standard'] = [rmsle_train, rmsle_valid]
df_res = pd.DataFrame.from_dict(res, orient='index',
                                columns=['rmsle_train', 'rmsle_valid'])
df_res = df_res.sort_values('rmsle_valid')
print(df_res)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.966e+05, tolerance: 2.188e+05
  model = cd_fast.enet_coordinate_descent(


Train RMSLE: 0.296641
Val RMSLE:   0.294904
                      rmsle_train  rmsle_valid
ElasticNet_MinMax       -0.898406    -0.898806
Lasso_Robust            -0.576662    -0.577741
Lasso_Standard          -0.573146    -0.573873
Ridge_Standard          -0.564620    -0.567380
Ridge_Robust            -0.564620    -0.567379
Ridge_MinMax            -0.564590    -0.567345
Lasso_MinMax            -0.469986    -0.468928
ElasticNet_Standard     -0.448930    -0.448114
ElasticNet_Robust       -0.446483    -0.446361
Ridge_poly2_standard     0.296641     0.294904


**Опиши, как ты подбирал полиномы здесь:**

перебрал полином 2 и 3 степени (2 оказался лучше)

лассо оказалась лучшей

для ОНЕ не применяется так как бинарные
